# Évaluation des modèles sur la validation

Comparer les modèles à partir de leurs artefacts sauvegardés.

| Métrique | Interprétation |
|---|---|
| F1 macro | Moyenne des F1 des huit tags, avec un poids égal. Critère principal.
| F1 micro | F1 calculé en regroupant toutes les décisions sur les tags.
| Exact Match Ratio | Proportion d’exercices dont les huit décisions sont toutes correctes. 
| Hamming Loss | Proportion de décisions binaires incorrectes parmi tous les exercices et tags. 
| Inférence médiane (ms/exercice) | Médiane des durées unitaires sur toute la validation. 

Chaque exercice est prédit individuellement. Le chronométrage comprend la préparation, la tokenisation/vectorisation, l’encodage, la classification et les seuils ; il exclut le chargement de l’artefact et le calcul des métriques. Ces performances restent des résultats sur la validation ayant servi aux réglages.

## 1. Charger la validation

L’ordre des huit tags est commun à tous les modèles. Les versions affichées et les
empreintes disponibles dans les artefacts permettent de contrôler le contexte de
l’évaluation. Les modèles sont chargés successivement pour limiter la mémoire.

In [2]:
import gc
import hashlib
import html
import json
import platform
import unicodedata
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import AutoTokenizer, BertConfig, BertModel
from scipy.sparse import hstack
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, f1_score, hamming_loss
from threadpoolctl import threadpool_limits
from IPython.display import display

racine = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "data" / "validation.jsonl").is_file()), None)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

torch.set_num_threads(4)
limite_threads = threadpool_limits(limits=1, user_api="blas")

tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
contenu_validation = (racine / "data" / "validation.jsonl").read_bytes()
empreinte_validation = hashlib.sha256(contenu_validation).hexdigest()
validation = pd.DataFrame([json.loads(ligne) for ligne in
    contenu_validation.decode("utf-8").splitlines() if ligne.strip()])
assert len(validation) > 0 and validation[colonnes_cibles].isin([0, 1]).all().all()
assert validation["fichier_source"].is_unique
assert validation["source_code"].map(lambda x: isinstance(x, str) and bool(x.strip())).all()
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
exercices = validation.to_dict("records")
resultats = []
print(f"{len(validation)} exercices de validation.")

739 exercices de validation.


## 2. Réutiliser les modèles sauvegardés

Les fonctions ci-dessous reproduisent la préparation et les représentations des
notebooks d’entraînement : quatre sections pour l’énoncé, code source inchangé,
blocs de tokens pour MiniLM/UniXcoder et fusion pour M9/M10. M11 est reconstruit depuis sa configuration locale et ses poids PyTorch, avec son tokenizer local et sa limite sauvegardée.

In [3]:
def construire_enonce(exercice, champs):
    sections = []
    for champ, titre in champs.items():
        valeur = exercice.get(champ)
        if isinstance(valeur, str) and valeur.strip():
            contenu = unicodedata.normalize("NFC", html.unescape(valeur)).strip()
            sections.append(f"{titre}\n{contenu}")
    return "\n\n".join(sections)


class ClassifieurBertMini(nn.Module):
    def __init__(self, configuration_bert, dropout, nombre_tags):
        super().__init__()
        self.bert = BertModel(configuration_bert, add_pooling_layer=False)
        self.dropout = nn.Dropout(dropout)
        self.tete = nn.Linear(2 * configuration_bert.hidden_size, nombre_tags)

    def forward(self, enonce, code):
        vecteur_enonce = self.bert(**enonce).last_hidden_state[:, 0]
        vecteur_code = self.bert(**code).last_hidden_state[:, 0]
        return self.tete(self.dropout(torch.cat([vecteur_enonce, vecteur_code], dim=1)))


def charger_modele(dossier, identifiant):
    dossier = racine / "models" / dossier
    if identifiant == "M11":
        informations = json.loads((dossier / "configuration.json").read_text(encoding="utf-8"))
        config = BertConfig.from_dict(informations["configuration_bert"])
        config._attn_implementation = informations["architecture"]["attention"]
        reseau = ClassifieurBertMini(config, informations["architecture"]["dropout"],
                                    len(informations["tags"]))
        reseau.load_state_dict(torch.load(dossier / "modele.pt", map_location="cpu",
                                          weights_only=True), strict=True)
        modele = {**informations, "reseau": reseau.to("cpu").eval(),
                  "tokenizer": AutoTokenizer.from_pretrained(
                      dossier / "tokenizer", local_files_only=True)}
    else:
        modele = joblib.load(dossier / "modele.joblib")
        if "encodeur" in modele:
            modele["encodeur"].to("cpu").eval()
    assert modele["modele"] == identifiant, f"Identifiant incorrect : {dossier.name}"
    assert modele["tags"] == tags_cibles, f"Ordre des tags incorrect : {identifiant}"
    assert np.shape(modele["seuils"]) == (8,) and np.isfinite(modele["seuils"]).all()
    empreintes = modele.get("empreintes_donnees", {})
    if "validation" in empreintes:
        assert empreintes["validation"] == empreinte_validation, \
            f"La validation diffère de celle du modèle {identifiant}."
    return modele

In [4]:
def decouper_textes(textes, encodeur, taille_bloc=254):
    contenus = encodeur.tokenizer(list(textes), add_special_tokens=False,
                                 truncation=False, padding=False, verbose=False)["input_ids"]
    if any(not contenu for contenu in contenus):
        raise ValueError("Un énoncé ne contient aucun token de contenu.")
    if not 1 <= taille_bloc <= encodeur.max_seq_length - 2:
        raise ValueError("La taille des blocs dépasse la capacité de MiniLM.")
    blocs, proprietaires, poids = [], [], []
    for indice, contenu in enumerate(contenus):
        for debut in range(0, len(contenu), taille_bloc):
            morceau = contenu[debut:debut + taille_bloc]
            ids = [
                encodeur.tokenizer.cls_token_id,
                *morceau,
                encodeur.tokenizer.sep_token_id
            ]

            blocs.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids)
            })
            proprietaires.append(indice)
            poids.append(len(morceau))
    longueurs = np.array([len(contenu) for contenu in contenus])
    proprietaires, poids = np.array(proprietaires), np.array(poids)
    np.testing.assert_array_equal(
        np.bincount(proprietaires, weights=poids, minlength=len(contenus)), longueurs
    )
    assert all(len(bloc["input_ids"]) <= 256 for bloc in blocs)
    return blocs, proprietaires, poids, longueurs


def encoder_textes(textes, encodeur, taille_bloc=254, taille_lot=16):
    blocs, proprietaires, poids, longueurs = decouper_textes(textes, encodeur, taille_bloc)
    vecteurs = np.zeros((len(longueurs), 384), dtype=np.float64)
    encodeur.eval()
    encodeur.tokenizer.deprecation_warnings["Asking-to-pad-a-fast-tokenizer"] = True
    with torch.inference_mode():
        for debut in range(0, len(blocs), taille_lot):
            fin = debut + taille_lot
            lot = encodeur.tokenizer.pad(blocs[debut:fin], padding=True, return_tensors="pt")
            lot = {cle: valeur.to(encodeur.device) for cle, valeur in lot.items()}
            embeddings = encodeur(lot)["sentence_embedding"].cpu().numpy()
            np.add.at(vecteurs, proprietaires[debut:fin], embeddings * poids[debut:fin, None])
    vecteurs /= longueurs[:, None]
    normes = np.linalg.norm(vecteurs, axis=1, keepdims=True)
    assert np.isfinite(vecteurs).all() and (normes > 0).all()
    vecteurs = (vecteurs / normes).astype(np.float32)
    return vecteurs, longueurs, np.bincount(proprietaires, minlength=len(longueurs))


def decouper_codes(codes, tokenizer, taille_bloc=508):
    if not 1 <= taille_bloc <= 508:
        raise ValueError("Les blocs doivent contenir entre 1 et 508 tokens de code.")
    if any(not isinstance(code, str) or not code.strip() for code in codes):
        raise ValueError("Chaque solution doit contenir un code non vide.")
    contenus = tokenizer(list(codes), add_special_tokens=False,
                         truncation=False, padding=False, verbose=False)["input_ids"]
    if not contenus or any(not contenu for contenu in contenus):
        raise ValueError("Un code ne contient aucun token.")
    prefixe = [tokenizer.cls_token_id,
               tokenizer.convert_tokens_to_ids("<encoder-only>"), tokenizer.sep_token_id]
    blocs, proprietaires, poids = [], [], []
    for indice, contenu in enumerate(contenus):
        for debut in range(0, len(contenu), taille_bloc):
            morceau = contenu[debut:debut + taille_bloc]
            ids = prefixe + morceau + [tokenizer.sep_token_id]
            blocs.append({"input_ids": ids, "attention_mask": [1] * len(ids)})
            proprietaires.append(indice)
            poids.append(len(morceau))
    longueurs = np.array([len(contenu) for contenu in contenus])
    proprietaires, poids = np.array(proprietaires), np.array(poids)
    np.testing.assert_array_equal(
        np.bincount(proprietaires, weights=poids, minlength=len(contenus)), longueurs
    )
    assert all(len(bloc["input_ids"]) <= 512 for bloc in blocs)
    return blocs, proprietaires, poids, longueurs


def encoder_codes(codes, encodeur, tokenizer, taille_bloc=508, taille_lot=8):
    if taille_lot < 1:
        raise ValueError("La taille des lots doit être positive.")
    blocs, proprietaires, poids, longueurs = decouper_codes(codes, tokenizer, taille_bloc)
    ordre = np.argsort([len(bloc["input_ids"]) for bloc in blocs], kind="stable")
    blocs = [blocs[indice] for indice in ordre]
    proprietaires, poids = proprietaires[ordre], poids[ordre]
    vecteurs = np.zeros((len(longueurs), encodeur.config.hidden_size), dtype=np.float64)
    encodeur.eval()
    appareil = next(encodeur.parameters()).device
    with torch.inference_mode():
        for debut in range(0, len(blocs), taille_lot):
            fin = debut + taille_lot
            lot = tokenizer.pad(blocs[debut:fin], padding=True, return_tensors="pt")
            ids = lot["input_ids"].to(appareil)
            masque = lot["attention_mask"].to(appareil)
            sorties = encodeur(input_ids=ids, attention_mask=masque).last_hidden_state
            embeddings = (sorties * masque.unsqueeze(-1)).sum(1) / masque.sum(1, keepdim=True)
            embeddings = embeddings.cpu().numpy()
            np.add.at(vecteurs, proprietaires[debut:fin], embeddings * poids[debut:fin, None])
    vecteurs /= longueurs[:, None]
    normes = np.linalg.norm(vecteurs, axis=1, keepdims=True)
    assert np.isfinite(vecteurs).all() and (normes > 0).all()
    vecteurs = (vecteurs / normes).astype(np.float32)
    return vecteurs, longueurs, np.bincount(proprietaires, minlength=len(longueurs))

In [5]:
def predire_exercice(exercice, modele):
    if modele["modele"] == "M11":
        textes = {"enonce": construire_enonce(exercice, modele["champs_texte"]),
                  "code": exercice[modele["champ_code"]]}
        entrees = {source: modele["tokenizer"](
            [texte], max_length=modele["tokenisation"]["longueur_max"],
            truncation=True, padding=True, return_tensors="pt")
            for source, texte in textes.items()}
        with torch.inference_mode():
            scores = torch.sigmoid(modele["reseau"](**entrees))[0].numpy()
    else:
        if "vectoriseur_enonce" in modele:
            enonce = construire_enonce(exercice, modele["champs_texte"])
            blocs = [modele["vectoriseur_enonce"].transform([enonce]),
                     modele["vectoriseur_code"].transform([exercice[modele["champ_code"]]])]
            matrice = normalize(hstack(blocs, format="csr"), norm="l2", copy=False)
        elif "vectoriseur" in modele:
            texte = (construire_enonce(exercice, modele["champs_texte"])
                     if "champs_texte" in modele else exercice[modele["champ_code"]])
            matrice = modele["vectoriseur"].transform([texte])
        else:
            parametres = {k: modele["encodage"][k] for k in ["taille_bloc", "taille_lot"]}
            if "champs_texte" in modele:
                texte = construire_enonce(exercice, modele["champs_texte"])
                matrice = encoder_textes([texte], modele["encodeur"], **parametres)[0]
            else:
                matrice = encoder_codes([exercice[modele["champ_code"]]],
                    modele["encodeur"], modele["tokenizer"], **parametres)[0]
        classifieur = modele["classifieur"]
        scores = (classifieur.predict_proba(matrice)[0]
                  if modele["type_scores"] == "probabilites"
                  else classifieur.decision_function(matrice)[0])
    if np.shape(scores) != (8,) or not np.isfinite(scores).all():
        raise ValueError("Le modèle doit produire huit scores finis.")
    return (scores >= np.asarray(modele["seuils"])).astype(np.int64)


def evaluer_groupe(modeles):
    lignes = []
    for identifiant, dossier in modeles:
        print(f"{identifiant} — chargement et évaluation sur {len(exercices)} exercices…", flush=True)
        modele = charger_modele(dossier, identifiant)
        for exercice in exercices[:3]:
            predire_exercice(exercice, modele)  
        predictions, durees = [], []
        for numero, exercice in enumerate(exercices, 1):
            debut = perf_counter()
            prediction = predire_exercice(exercice, modele)
            durees.append(1000 * (perf_counter() - debut))
            predictions.append(prediction)
        predictions = np.stack(predictions)
        ligne = {"Modèle": identifiant,
                 "F1 macro": f1_score(y_validation, predictions, average="macro", zero_division=0),
                 "F1 micro": f1_score(y_validation, predictions, average="micro", zero_division=0),
                 "Exact Match Ratio": accuracy_score(y_validation, predictions),
                 "Hamming Loss": hamming_loss(y_validation, predictions),
                 "Inférence médiane (ms/exercice)": float(np.median(durees))}
        assert len(durees) == len(validation) and predictions.shape == y_validation.shape
        lignes.append(ligne)
        resultats.append(ligne)
        del modele
        gc.collect()
    display(pd.DataFrame(lignes).round(4))

## 3. Références

**Aucun tag** prédit huit zéros pour chaque exercice ; **Tous les tags**, huit uns. Ces deux règles constantes permettent de situer les modèles. Les deux approches sont évaluées sur toute la validation. Pour les règles constantes, le temps d’inférence mesure uniquement la production des huit décisions, sans traitement du texte.

In [6]:
lignes_references = []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions, durees = [], []
    for exercice in exercices:
        debut = perf_counter()
        prediction = np.full(len(tags_cibles), valeur, dtype=np.int64)
        durees.append(1000 * (perf_counter() - debut))
        predictions.append(prediction)
    predictions = np.stack(predictions)
    assert predictions.shape == y_validation.shape and len(durees) == len(validation)
    lignes_references.append({
        "Modèle": nom,
        "F1 macro": f1_score(y_validation, predictions, average="macro", zero_division=0),
        "F1 micro": f1_score(y_validation, predictions, average="micro", zero_division=0),
        "Exact Match Ratio": accuracy_score(y_validation, predictions),
        "Hamming Loss": hamming_loss(y_validation, predictions),
        "Inférence médiane (ms/exercice)": float(np.median(durees)),
    })
comparaison_references = pd.DataFrame(lignes_references)
display(comparaison_references.round(4))

,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,Aucun tag,0.00,0.0000,0.4587,0.0864,0.0011
1,Tous les tags,0.15,0.1591,0.0000,0.9136,0.0007


## 4. Énoncé — Word TF-IDF : M0, M1 et M2

Les mots et paires de mots des quatre sections de l’énoncé constituent les variables. **M0** utilise Naïve Bayes multinomial comme référence ; **M1**, une régression logistique ; **M2**, un SVM linéaire. Les trois modèles partagent le même Word TF-IDF.

In [7]:
evaluer_groupe([('M0', 'm0_word_tfidf_naive_bayes'), ('M1', 'm1_word_tfidf_logistic_regression'), ('M2', 'm2_word_tfidf_linear_svm')])

M0 — chargement et évaluation sur 739 exercices…
M1 — chargement et évaluation sur 739 exercices…
M2 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M0,0.6161,0.5763,0.4425,0.0846,2.7638
1,M1,0.6659,0.6293,0.5575,0.0660,0.6564
2,M2,0.6709,0.6320,0.5643,0.0646,0.5671


## 5. Énoncé — MiniLM : M3 et M4

**M3** utilise une régression logistique et **M4** un SVM linéaire sur les vecteurs
MiniLM. Les énoncés sont découpés en blocs de 254 tokens de contenu, leurs vecteurs
sont moyennés avec une pondération par longueur, puis normalisés en L2. L’encodeur reste figé.

In [8]:
evaluer_groupe([('M3', 'm3_minilm_logistic_regression'), ('M4', 'm4_minilm_linear_svm')])

M3 — chargement et évaluation sur 739 exercices…
M4 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M3,0.6246,0.5771,0.4357,0.0868,61.9287
1,M4,0.5788,0.5471,0.4046,0.0935,62.9553


## 6. Code — TF-IDF : M5 et M6

Les identifiants, nombres, opérateurs et signes du code de la solution sont représentés par
un TF-IDF de tokens et paires de tokens. **M5** utilise une régression logistique ; **M6**, un SVM linéaire.

In [9]:
evaluer_groupe([('M5', 'm5_code_tfidf_logistic_regression'), ('M6', 'm6_code_tfidf_linear_svm')])

M5 — chargement et évaluation sur 739 exercices…
M6 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M5,0.4477,0.5184,0.4357,0.0952,0.5219
1,M6,0.4493,0.5219,0.4384,0.0942,0.4547


## 7. Code — UniXcoder : M7 et M8

**M7** utilise une régression logistique et **M8** un SVM linéaire sur les vecteurs
UniXcoder. Les codes sont découpés en blocs de 508 tokens de contenu et les vecteurs sont moyennés avec une pondération par longueur puis normalisés en L2. L’encodeur reste figé.

In [10]:
evaluer_groupe([('M7', 'm7_unixcoder_logistic_regression'), ('M8', 'm8_unixcoder_linear_svm')])

M7 — chargement et évaluation sur 739 exercices…
M8 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M7,0.4082,0.4928,0.4736,0.0888,119.8940
1,M8,0.4002,0.4839,0.3897,0.1032,115.5915


## 8. Énoncé + Code — TF-IDF : M9 et M10

Les TF-IDF de l’énoncé et du code sont normalisés en L2, concaténés avec le même
poids, puis normalisés en L2. **M9** applique une régression logistique ; **M10**,
un SVM linéaire. Les deux sources sont utilisées simultanément.

In [11]:
evaluer_groupe([('M9', 'm9_fusion_tfidf_logistic_regression'), ('M10', 'm10_fusion_tfidf_linear_svm')])

M9 — chargement et évaluation sur 739 exercices…
M10 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M9,0.6748,0.6376,0.5535,0.0646,0.9839
1,M10,0.6777,0.6425,0.5562,0.0650,0.9298


## 9. Énoncé + Code — BERT-mini fine-tuné : M11

**M11** effectue deux passages dans un BERT-mini partagé, un pour chaque source
limitée à 512 tokens. Les deux vecteurs sont concaténés et une tête linéaire
produit huit logits. La dernière couche et la tête ont été entraînées. La sigmoid
et les huit seuils sauvegardés déterminent les tags. Seuls les poids retenus sont rechargés.

In [12]:
evaluer_groupe([('M11', 'm11_bert_mini_finetuning')])

M11 — chargement et évaluation sur 739 exercices…


,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M11,0.5839,0.5913,0.5129,0.0734,29.9555


## 10. Classement récapitulatif

Classer M0 à M11 par **F1 macro décroissant**, le critère principal. Les cinq métriques proviennent des évaluations précédentes sur toute la validation, aucune prédiction n’est relancée.

In [13]:
classement = pd.DataFrame(resultats)
assert set(classement["Modèle"]) == {f"M{i}" for i in range(12)} and classement["Modèle"].is_unique, \
    "Exécutez les comparaisons des douze modèles dans l’ordre avant cette section."
classement["numero"] = classement["Modèle"].str[1:].astype(int)
classement = classement.sort_values(
    ["F1 macro", "numero"], ascending=[False, True]
).drop(columns="numero").reset_index(drop=True)
classement.index = np.arange(1, len(classement) + 1)
classement.index.name = "Rang"
display(classement.round(4))

,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
Rang,,,,,,
1,M10,0.6777,0.6425,0.5562,0.0650,0.9298
2,M9,0.6748,0.6376,0.5535,0.0646,0.9839
3,M2,0.6709,0.6320,0.5643,0.0646,0.5671
4,M1,0.6659,0.6293,0.5575,0.0660,0.6564
5,M3,0.6246,0.5771,0.4357,0.0868,61.9287
6,M0,0.6161,0.5763,0.4425,0.0846,2.7638
7,M11,0.5839,0.5913,0.5129,0.0734,29.9555
8,M4,0.5788,0.5471,0.4046,0.0935,62.9553
9,M6,0.4493,0.5219,0.4384,0.0942,0.4547


## 11. Modèle retenu

Les trois premiers modèles ont des F1 macro proches sur validation : 0,678 pour **M10**,
0,675 pour **M9** et 0,671 pour **M2**. Nous retenons **M2, Word TF-IDF + SVM linéaire**,
comme compromis entre performances et simplicité.

M2 utilise uniquement les quatre sections de l’énoncé, avec un seul vectoriseur.
M9/M10 nécessitent aussi le code source et un second TF-IDF. M2 réduit ainsi les
entrées et les étapes de traitement, simplifie le déploiement et présente une
inférence plus rapide dans nos mesures. Son Exact Match Ratio est le meilleur
des trois : 0,564, contre 0,553 pour M9 et 0,556 pour M10.

Le gain de F1 macro des fusions reste limité et sa robustesse statistique n’a pas
été établie. Le choix de M2 privilégie donc une solution plus parcimonieuse, il
ne suppose pas une supériorité démontrée en généralisation.

Recharger M2 avec ses seuils sauvegardés, recalculer les cinq métriques sur toute
la validation, puis détailler précision, rappel, F1 et support par tag. Ce bilan
reste une analyse sur validation, le test est réservé à l’évaluation finale.

In [16]:
from sklearn.metrics import precision_recall_fscore_support

assert {ligne["Modèle"] for ligne in resultats} == {f"M{i}" for i in range(12)}, \
    "Exécutez les comparaisons des douze modèles avant cette section."

identifiant_retenu = "M2"
modele_retenu = charger_modele("m2_word_tfidf_linear_svm", identifiant_retenu)

for exercice in exercices[:3]:
    predire_exercice(exercice, modele_retenu)
predictions_retenues, durees_retenues = [], []
for exercice in exercices:
    debut = perf_counter()
    prediction = predire_exercice(exercice, modele_retenu)
    durees_retenues.append(1000 * (perf_counter() - debut))
    predictions_retenues.append(prediction)
predictions_retenues = np.stack(predictions_retenues)
assert predictions_retenues.shape == y_validation.shape and len(durees_retenues) == len(validation)

bilan_retenu = pd.DataFrame([{
    "Modèle": identifiant_retenu,
    "F1 macro": f1_score(y_validation, predictions_retenues, average="macro", zero_division=0),
    "F1 micro": f1_score(y_validation, predictions_retenues, average="micro", zero_division=0),
    "Exact Match Ratio": accuracy_score(y_validation, predictions_retenues),
    "Hamming Loss": hamming_loss(y_validation, predictions_retenues),
    "Inférence médiane (ms/exercice)": float(np.median(durees_retenues)),
}])
display(bilan_retenu.round(4))

precisions, rappels, f1_tags, supports = precision_recall_fscore_support(
    y_validation, predictions_retenues, average=None, zero_division=0)
bilan_par_tag = pd.DataFrame({
    "Tag": tags_cibles, "Support": supports.astype(int), "Précision": precisions, "Rappel": rappels,
    "F1": f1_tags,
})
display(bilan_par_tag.round(4))

,Modèle,F1 macro,F1 micro,Exact Match Ratio,Hamming Loss,Inférence médiane (ms/exercice)
0,M2,0.6709,0.632,0.5643,0.0646,0.5679


,Tag,Support,Précision,Rappel,F1
0,math,211,0.5570,0.6256,0.5893
1,graphs,81,0.6543,0.6543,0.6543
2,strings,63,0.6479,0.7302,0.6866
3,number theory,52,0.5893,0.6346,0.6111
4,trees,49,0.7838,0.5918,0.6744
5,geometry,25,0.6154,0.6400,0.6275
6,games,16,1.0000,0.7500,0.8571
7,probabilities,14,1.0000,0.5000,0.6667
